# Hair App Canonical Crop V3 — Five-Point Roll

v3는 roll을 두 눈의 선 하나로 정하지 않는다. **두 눈·코끝·두 입꼬리의 5점 도형 전체**를 코끝 기준의 정방향 도형에 가장 잘 맞추는 단일 회전을 계산한다.

얼굴을 비틀거나 yaw/pitch를 없애지 않고 사진 전체에는 회전 한 번만 적용한다. Pixel3DMM·segmentation은 아직 실행하지 않는다.

In [ ]:
!pip -q install 'git+https://github.com/FacePerceiver/facer.git@ddd35c76ff840174b8a5403ad1c1255e37b8782b'
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Drive와 경로

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
INPUT_DIR = Path('/content/drive/MyDrive/hair_app/inputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/hair_app/crop_test_512_v3')
OUTPUT_SIZE = 512
BBOX_MARGIN = 1.50
VERTICAL_CENTER_OFFSET = -0.04
DETECTION_THRESHOLD = 0.5
input_files = sorted(path for path in INPUT_DIR.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print('inputs:', len(input_files), [path.name for path in input_files])
assert input_files, f'입력 이미지 없음: {INPUT_DIR}'

## 2. v2 공통 코드와 v3 엔진 준비

아직 GitHub push 전이라면 파일 선택 창에서 `canonical_face_crop_v2.py`와 `canonical_face_crop_v3.py` 두 파일을 함께 선택한다.

In [ ]:
from urllib.request import urlretrieve
from google.colab import files

engine_specs = {
    'canonical_face_crop_v2.py': 'https://raw.githubusercontent.com/Leejuseop/hair_app/main/experiments/milestone1_geometry_bakeoff/canonical_face_crop_v2.py',
    'canonical_face_crop_v3.py': 'https://raw.githubusercontent.com/Leejuseop/hair_app/main/experiments/milestone1_geometry_bakeoff/canonical_face_crop_v3.py',
}
missing = []
for name, url in engine_specs.items():
    try:
        urlretrieve(url, Path('/content') / name)
        print(name, 'GitHub download: PASS')
    except Exception as error:
        print(name, 'GitHub download 실패:', error)
        missing.append(name)
if missing:
    print('다음 파일을 함께 선택하세요:', missing)
    uploaded = files.upload()
    for name in missing:
        assert name in uploaded, f'누락: {name}'
        (Path('/content') / name).write_bytes(uploaded[name])
for name in engine_specs:
    path = Path('/content') / name
    assert path.exists() and path.stat().st_size > 0
print('V3 ENGINE FILES: PASS')

## 3. v3 crop 실행

In [ ]:
import shutil
import subprocess
import sys

assert OUTPUT_ROOT.name == 'crop_test_512_v3', f'안전하지 않은 출력 경로: {OUTPUT_ROOT}'
shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)
command = [
    sys.executable, '/content/canonical_face_crop_v3.py',
    '--input-dir', str(INPUT_DIR),
    '--output-root', str(OUTPUT_ROOT),
    '--output-size', str(OUTPUT_SIZE),
    '--bbox-margin', str(BBOX_MARGIN),
    '--vertical-center-offset', str(VERTICAL_CENTER_OFFSET),
    '--detection-threshold', str(DETECTION_THRESHOLD),
    '--device', DEVICE,
    '--clean-output',
]
subprocess.run(command, check=True)
print('CROP V3 RUN: PASS')

## 4. 원본과 v3 비교

청록색은 눈, 노란색은 코끝, 자홍색은 입꼬리다. 제목에 `eye`는 기존 눈선 roll, `five`는 v3가 5점 전체로 계산해 실제 적용한 roll이다.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image, ImageOps

manifest = json.loads((OUTPUT_ROOT / 'crop_meta' / 'manifest.json').read_text(encoding='utf-8'))
fig, axes = plt.subplots(len(manifest['items']), 3, figsize=(15, 4.5 * len(manifest['items'])), squeeze=False)
for row, item in enumerate(manifest['items']):
    with Image.open(INPUT_DIR / item['source_name']) as source_file:
        source = ImageOps.exif_transpose(source_file).convert('RGB')
    crop = Image.open(OUTPUT_ROOT / 'cropped' / item['derived_name']).convert('RGB')
    validity = Image.open(OUTPUT_ROOT / 'crop_validity' / item['validity_mask_name']).convert('L')
    points = np.asarray(item['observation']['landmarks5_xy'])

    axes[row, 0].imshow(source)
    selected = item['face_selection']['selected_detector_index']
    for candidate in item['face_selection']['candidate_rankings']:
        x1, y1, x2, y2 = candidate['bbox_xyxy']
        color = 'lime' if candidate['detector_index'] == selected else 'red'
        axes[row, 0].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color=color, linewidth=2))
    axes[row, 0].plot(points[:2, 0], points[:2, 1], color='cyan', linewidth=2)
    axes[row, 0].plot(points[3:, 0], points[3:, 1], color='magenta', linewidth=2)
    axes[row, 0].scatter(points[:, 0], points[:, 1], s=28, c=['cyan','cyan','yellow','magenta','magenta'])
    axes[row, 0].set_title(f"원본: {item['source_name']}")

    transformed = np.asarray(item['transformed_landmarks5'])
    axes[row, 1].imshow(crop)
    axes[row, 1].plot(transformed[:2, 0], transformed[:2, 1], color='cyan', linewidth=2)
    axes[row, 1].plot(transformed[3:, 0], transformed[3:, 1], color='magenta', linewidth=2)
    axes[row, 1].scatter(transformed[:, 0], transformed[:, 1], s=28, c=['cyan','cyan','yellow','magenta','magenta'])
    components = item['five_point_fit']['component_roll_degrees']
    axes[row, 1].set_title(
        f"v3: eye={components['eye_line']:.1f}° / five={item['roll_degrees_applied']:.1f}° / fit={item['five_point_fit']['normalized_fit_residual']:.3f}\n"
        f"warnings={item['warnings']}"
    )

    axes[row, 2].imshow(validity, cmap='gray', vmin=0, vmax=255)
    axes[row, 2].set_title(f"observed source={item['observed_source_fraction']:.3f}")
    for axis in axes[row]:
        axis.axis('off')
plt.tight_layout()
plt.show()

## 5. 자동 계약 검사

In [ ]:
crop_files = sorted((OUTPUT_ROOT / 'cropped').glob('*.jpg'))
mask_files = sorted((OUTPUT_ROOT / 'crop_validity').glob('*.png'))
meta_files = sorted(path for path in (OUTPUT_ROOT / 'crop_meta').glob('*.json') if path.name != 'manifest.json')
assert len(input_files) == len(crop_files) == len(mask_files) == len(meta_files) == manifest['count']
for crop_path, mask_path, item in zip(crop_files, mask_files, manifest['items']):
    assert Image.open(crop_path).size == (512, 512)
    assert Image.open(mask_path).size == (512, 512)
    forward = np.asarray(item['source_to_crop'])
    inverse = np.asarray(item['crop_to_source'])
    assert np.allclose(inverse @ forward, np.eye(3), atol=1e-6)
    assert item['roll_method'] == 'nose_anchored_five_point_similarity'
print('CROP V3 CONTRACT: PASS')
print('위 결과에서 특히 기존 24.6° 사진과 두 profile 사진의 eye/five 값을 비교하세요.')